In [1]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve )

In [2]:
# Read in book and movie review data
books = pd.read_csv("data/df_book_reviews_spoiler_features.csv")
books = books.reset_index(drop=True)
movies = pd.read_csv("data/df_movie_reviews_spoiler_features.csv")
movies = movies.reset_index(drop=True) 

In [3]:
# Load in tf-idf datasets
import scipy.sparse as sp
Xm = sp.load_npz("data/movie_X_tfidf.npz")
Xb = sp.load_npz("data/book_X_tfidf.npz")

In [4]:
# Save feature list for each dataset
common_features = [
    "log_word_count",
    "log_sentence_count",
    "log_avg_sentence_length",
    "capitalized_word_density",
    "log_third_first_pronoun_ratio",
    "log_quote_count",
    "log_exclamation_count",
    "log_question_count",
    "log_number_count",
    "spoiler_keyword_count",
    "spoiler_keyword_flag"
]

movie_features = [
    "movie_specificity_mean",
    "movie_specificity_max"
] + common_features

book_features = [
    "book_specificity_mean",
    "book_specificity_max"
] + common_features

In [5]:
# Function to create 70/15/15 train, validation and test data split
def split_data(df, id_col):
    """Create grouped 70/15/15 train, validation, and test splits."""

    gss = GroupShuffleSplit(
        n_splits=1, test_size=0.30, random_state=42
    )

    train_idx, rest_idx = next(
        gss.split(df, groups=df[id_col])
    )

    rest = df.iloc[rest_idx]

    gss2 = GroupShuffleSplit(
        n_splits=1, test_size=0.50, random_state=42
    )

    val_i, test_i = next(
        gss2.split(rest, groups=rest[id_col])
    )

    val_idx = rest_idx[val_i]
    test_idx = rest_idx[test_i]

    return train_idx, val_idx, test_idx

In [6]:
# Function to final optimal classification threshold (w/ validation set)
def best_f1(y_true, p):
    """Find the classification threshold that maximizes F1."""

    prec, rec, thr = precision_recall_curve(y_true, p)

    f1 = (
        2 * prec[:-1] * rec[:-1]
        / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    )

    i = f1.argmax()

    return f1[i], thr[i]

In [7]:
# Function to train and evaluate LR model
def run_logistic_regression(df, X, id_col, feat_cols, name):
    
    df = df.reset_index(drop=True)
    assert X.shape[0] == len(df), "Row mismatch"

    y = df["has_spoiler"].to_numpy()

    # Create data splits
    train_idx, val_idx, test_idx = split_data(df, id_col)

    # Scale engineered features using training data only
    scaler = StandardScaler()
    scaler.fit(df.loc[train_idx, feat_cols])

    F = scaler.transform(df[feat_cols]).astype("float32")

    # Combine TF-IDF and engineered features
    X2 = sp.hstack([X, sp.csr_matrix(F)], format="csr")

    # Train logistic regression, keeping time
    t = time.perf_counter()

    model = LogisticRegression(
        C=1,
        class_weight="balanced",
        solver="liblinear"
    )

    model.fit(X2[train_idx], y[train_idx])

    fit_seconds = time.perf_counter() - t

    # Determine optimal threshold using validation data
    pv = model.predict_proba(X2[val_idx])[:, 1]
    f1_val, thr = best_f1(y[val_idx], pv)

    # Generate test predictions
    t = time.perf_counter()

    pt = model.predict_proba(X2[test_idx])[:, 1]

    predict_seconds = time.perf_counter() - t
    pred = pt >= thr

    # Calculate evaluation metrics
    results = {
        "Dataset": name,
        "Threshold": thr,
        "Validation F1": f1_val,
        "Precision": precision_score(y[test_idx], pred),
        "Recall": recall_score(y[test_idx], pred),
        "F1": f1_score(y[test_idx], pred),
        "ROC-AUC": roc_auc_score(y[test_idx], pt),
        "PR-AUC": average_precision_score(y[test_idx], pt),
        "Fit Seconds": fit_seconds,
        "Predict Seconds": predict_seconds
    }

    return results

In [8]:
# Run movie spoiler detection dataset
movie_results = run_logistic_regression(
    movies,
    Xm,
    "movie_id",
    movie_features,
    "Movies"
)

pd.DataFrame([movie_results]).round(4)

,Dataset,Threshold,Validation F1,Precision,Recall,F1,ROC-AUC,PR-AUC,Fit Seconds,Predict Seconds
0,Movies,0.4936,0.5612,0.4876,0.6629,0.5619,0.7804,0.588,110.6026,0.1274


In [9]:
# Run books spoiler detection dataset
book_results = run_logistic_regression(
    books,
    Xb,
    "book_id",
    book_features,
    "Books"
)

pd.DataFrame([book_results]).round(4)

,Dataset,Threshold,Validation F1,Precision,Recall,F1,ROC-AUC,PR-AUC,Fit Seconds,Predict Seconds
0,Books,0.6994,0.3896,0.3291,0.4701,0.3872,0.855,0.3444,303.3927,0.3266
